# F1 Textbook: Bylaw Intelligence Command Center

This notebook is the living textbook for F1. It is written for an active learner who understands the basic idea of AI/RAG but has not executed this project before.

After reading and running it, you should be able to explain:

- why F1 exists after the M-series,
- how source documents become verified rules,
- why verification is the authority,
- what each layer receives and sends forward,
- how to run checkpoint 0,
- how to judge whether the system is safe enough to upgrade.

## 1. Big Picture

F1 is not just a bylaw chatbot. It is a **Bylaw Intelligence Command Center**.

The product question is:

> Given this bylaw document and this project scenario, what is allowed, what is restricted, what evidence supports it, what is uncertain, and what should I do next?

The most important rule is simple:

```text
AI proposes. Retrieval supports. Verification decides. Humans review uncertainty.
```

## 2. F1 Pipeline

```mermaid
flowchart LR
    A[Official bylaw PDF] --> B[Source registry]
    B --> C[Document intelligence]
    C --> D[Proof-aware RAG]
    D --> E[Structured extraction]
    E --> F[Deterministic verification]
    F --> G[Verified rules]
    F --> H[Review queue]
    H --> I[Repair proposals]
    I --> F
    G --> J[GIS and compliance]
    G --> K[Command center chatbot]
    H --> K
```

A useful mental model:

| Layer | It does | It cannot do |
|---|---|---|
| Source registry | records official sources | decide rules |
| Document intelligence | parses PDFs, tables, drawings | prove legal truth |
| RAG | finds evidence | verify rules |
| Extraction | proposes candidate rules | promote rules |
| Verification | proves or rejects support | invent missing evidence |
| Repair | proposes proof patches | approve rules |
| Chatbot | explains and navigates | override verifier |
| GIS/compliance | uses verified rules | consume review rules as verified |

## 3. Why The M-Series Matters

The M-series was the research path. Its most important lesson was not a model name or a prompt. The lesson was the trust boundary:

```text
Extraction proposes.
RAG retrieves source context.
Repair improves evidence.
Deterministic verification decides.
GIS consumes verified-only rules.
```

Checkpoint 0 keeps that safety boundary while creating a cleaner product structure for F1.

Recent measured M8.5 lessons to remember:

| Lesson | Meaning for F1 |
|---|---|
| Burnaby repair improved when exact table proof existed | table lineage matters |
| Calgary and Vancouver ran safely but repair did not generalize enough | repair must be evidence-format aware |
| Surrey without gold benchmark is exploratory only | no gold means no release-safe claim |
| Dry-run LLM repair is diagnostic only | real API results must be labeled separately |

## 4. Inspect The F1 Contracts

Run the next cell from the `Burnaby_prototype` folder. It loads the F1 contract layer and prints the canonical artifact names.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT / 'src'))

from burnaby_prototype.bylaw_command_center import CANONICAL_ARTIFACT_NAMES, ARTIFACT_SCHEMA_VERSION

print('Schema:', ARTIFACT_SCHEMA_VERSION)
print('Canonical artifacts:')
for name in CANONICAL_ARTIFACT_NAMES:
    print(' -', name)

## 5. Fail-Closed Behavior

F1 contracts should reject unsafe states early. A promoted verification decision must have evidence and cannot have support gaps, unresolved dependencies, or risk flags.

In [ ]:
from burnaby_prototype.bylaw_command_center import VerificationDecision, RepairPatch, ComplianceDecision

examples = []

try:
    VerificationDecision(rule_id='r1', status='verified')
except ValueError as exc:
    examples.append(('verified_without_evidence', str(exc)))

try:
    RepairPatch(patch_id='p1', rule_id='r1', action='mark_verified')
except ValueError as exc:
    examples.append(('repair_direct_promotion', str(exc)))

try:
    ComplianceDecision(scenario_id='s1', status='approved', review_rule_ids=('review_rule',))
except ValueError as exc:
    examples.append(('approval_depends_on_review_rule', str(exc)))

for name, message in examples:
    print(f'{name}: {message}')

## 6. Visual Trace

The next cell draws a simple SVG of the rule lifecycle. This is intentionally small enough to run anywhere.

In [ ]:
from IPython.display import SVG, display

svg = '''
<svg width='900' height='260' viewBox='0 0 900 260' xmlns='http://www.w3.org/2000/svg'>
  <style>
    .box{fill:#f6f8fa;stroke:#57606a;stroke-width:1.4;rx:8;ry:8}
    .good{fill:#dafbe1;stroke:#1a7f37}
    .warn{fill:#fff8c5;stroke:#9a6700}
    .bad{fill:#ffebe9;stroke:#cf222e}
    .txt{font:14px -apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;fill:#24292f}
    .small{font:12px -apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;fill:#57606a}
    .arrow{stroke:#57606a;stroke-width:1.6;marker-end:url(#arrow)}
  </style>
  <defs><marker id='arrow' markerWidth='8' markerHeight='8' refX='7' refY='4' orient='auto'><path d='M0,0 L8,4 L0,8 z' fill='#57606a'/></marker></defs>
  <rect class='box' x='25' y='80' width='120' height='70'/><text class='txt' x='48' y='108'>Source PDF</text><text class='small' x='48' y='130'>ground truth</text>
  <rect class='box' x='185' y='80' width='120' height='70'/><text class='txt' x='218' y='108'>RAG</text><text class='small' x='207' y='130'>finds proof</text>
  <rect class='box' x='345' y='80' width='120' height='70'/><text class='txt' x='374' y='108'>Extract</text><text class='small' x='368' y='130'>proposes</text>
  <rect class='box' x='505' y='80' width='120' height='70'/><text class='txt' x='535' y='108'>Verify</text><text class='small' x='520' y='130'>decides</text>
  <rect class='good' x='665' y='25' width='145' height='62'/><text class='txt' x='695' y='55'>Verified</text><text class='small' x='686' y='74'>GIS allowed</text>
  <rect class='warn' x='665' y='105' width='145' height='62'/><text class='txt' x='692' y='135'>Review</text><text class='small' x='681' y='154'>repair or human</text>
  <rect class='bad' x='665' y='185' width='145' height='50'/><text class='txt' x='694' y='216'>Rejected</text>
  <line class='arrow' x1='145' y1='115' x2='185' y2='115'/><line class='arrow' x1='305' y1='115' x2='345' y2='115'/><line class='arrow' x1='465' y1='115' x2='505' y2='115'/>
  <line class='arrow' x1='625' y1='105' x2='665' y2='58'/><line class='arrow' x1='625' y1='120' x2='665' y2='136'/><line class='arrow' x1='625' y1='135' x2='665' y2='210'/>
</svg>
'''
display(SVG(svg))

## 7. Run Checkpoint 0

Checkpoint 0 writes the first F1 summary and checks that the existing safety report still preserves the zero-false-verified boundary.

```bash
.venv/bin/python scripts/run_f1_checkpoint.py --city burnaby_r1 --run-id checkpoint_0
```

Expected outputs:

```text
outputs/f1_runs/burnaby_r1/checkpoint_0/f1_summary.json
outputs/f1_runs/burnaby_r1/checkpoint_0/checkpoint_report.md
```

In [ ]:
import json
from pathlib import Path

summary_path = Path('outputs/f1_runs/burnaby_r1/checkpoint_0/f1_summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print('Checkpoint:', summary['checkpoint'])
    print('Existing safety reproduced:', summary['safety']['existing_safety_reproduced'])
    print('False verified total:', summary['safety']['current_false_verified_total'])
else:
    print('Run the checkpoint command first, then rerun this cell.')

## 8. Run Checkpoint 1: Source Registry

Checkpoint 1 creates `source_registry.json`. This is the source provenance gate before document intelligence, RAG, extraction, verification, repair, chatbot, or GIS layers consume a bylaw PDF.

```bash
.venv/bin/python scripts/run_f1_checkpoint.py --checkpoint source_registry --city burnaby_r1
```

The registry passes only when the cached PDF exists, provenance exists, SHA-256 matches, and the config source URL matches provenance.

In [ ]:
source_registry_path = Path('outputs/f1_runs/burnaby_r1/checkpoint_1_source_registry/source_registry.json')
if source_registry_path.exists():
    registry = json.loads(source_registry_path.read_text())
    document = registry['documents'][0]
    print('Registry validation:', registry['validation']['status'])
    print('City:', registry['city'])
    print('Title:', document['title'])
    print('Hash verified:', document['hash_verified'])
    print('Page count:', document['page_count'])
else:
    print('Run checkpoint 1 first, then rerun this cell.')

## 9. Run Checkpoint 2: Document Intelligence

Checkpoint 2 parses the registered source PDF into `document_blocks.json` and `source_spans.json`.

```bash
.venv/bin/python scripts/run_f1_checkpoint.py --checkpoint document_intelligence --city burnaby_r1
```

The default path tries Docling first. If Docling is not available, use:

```bash
.venv/bin/python scripts/run_f1_checkpoint.py --checkpoint document_intelligence --city burnaby_r1 --no-docling
```

Important: document blocks and source spans are not verified rules. They are structured source material for later RAG, extraction, verification, and repair.

In [ ]:
doc_report_path = Path('outputs/f1_runs/burnaby_r1/checkpoint_2_document_intelligence/document_intelligence_report.json')
blocks_path = Path('outputs/f1_runs/burnaby_r1/checkpoint_2_document_intelligence/document_blocks.json')
if doc_report_path.exists() and blocks_path.exists():
    report = json.loads(doc_report_path.read_text())
    blocks = json.loads(blocks_path.read_text())
    print('Backend:', report['ingest_backend'])
    print('Summary:', report['summary'])
    print('\nFirst two blocks:')
    for block in blocks[:2]:
        print('-', block['block_type'], 'page', block['page'], block['text'][:120].replace('\n', ' '))
else:
    print('Run checkpoint 2 first, then rerun this cell.')

## 10. Run Checkpoint 3: Proof-Aware RAG

Checkpoint 3 runs deterministic retrieval over the F1 source spans and writes `retrieval_report.json`.

```bash
.venv/bin/python scripts/run_f1_checkpoint.py --checkpoint proof_rag --city burnaby_r1
```

The checkpoint 3 retriever uses lexical/BM25 reciprocal-rank fusion plus a lexical-overlap reranker. It is a smoke check, not legal verification.

In [ ]:
retrieval_path = Path('outputs/f1_runs/burnaby_r1/checkpoint_3_proof_rag/retrieval_report.json')
if retrieval_path.exists():
    retrieval = json.loads(retrieval_path.read_text())
    print('Summary:', retrieval['summary'])
    first_query = retrieval['queries'][0]
    print('\nFirst query:', first_query['query'])
    for hit in first_query['hits'][:3]:
        print(f"rank={hit['rank']} page={hit['page']} span={hit['span_id']} quote={hit['quote'][:120]}")
else:
    print('Run checkpoint 3 first, then rerun this cell.')

## 11. Run Checkpoint 4: Structured Extraction

Checkpoint 4 turns retrieval evidence into structured candidate rules with field citations.

```bash
.venv/bin/python scripts/run_f1_checkpoint.py --checkpoint structured_extraction --city burnaby_r1
```

Important: every checkpoint 4 candidate is still `review_needed`. The extractor proposes. It does not verify.

In [ ]:
candidate_path = Path('outputs/f1_runs/burnaby_r1/checkpoint_4_structured_extraction/candidate_rules.json')
extraction_report_path = Path('outputs/f1_runs/burnaby_r1/checkpoint_4_structured_extraction/extraction_report.json')
if candidate_path.exists() and extraction_report_path.exists():
    candidates = json.loads(candidate_path.read_text())
    extraction_report = json.loads(extraction_report_path.read_text())
    print('Extraction report:', extraction_report)
    print('\nCandidates:')
    for rule in candidates:
        print(rule['rule_id'], rule['rule_type'], rule['operator'], rule['value'], rule['unit'], rule['status'])
else:
    print('Run checkpoint 4 first, then rerun this cell.')

## 12. Run Checkpoint 5: Fail-Closed Verification

Checkpoint 5 runs deterministic verification over F1 candidate rules.

```bash
.venv/bin/python scripts/run_f1_checkpoint.py --checkpoint verification --city burnaby_r1
```

This checkpoint is expected to be conservative. Incomplete applicability or scope should go to `context_required`; unsupported evidence should be rejected; only fully supported rules can enter `verified_rules.json`.

In [ ]:
verification_report_path = Path('outputs/f1_runs/burnaby_r1/checkpoint_5_verification/verification_report.json')
decisions_path = Path('outputs/f1_runs/burnaby_r1/checkpoint_5_verification/verification_decisions.json')
if verification_report_path.exists() and decisions_path.exists():
    verification_report = json.loads(verification_report_path.read_text())
    decisions = json.loads(decisions_path.read_text())
    print('Verification report:', verification_report)
    print('\nDecisions:')
    for decision in decisions:
        print(decision['rule_id'], decision['status'], 'gaps=', decision['support_gaps'], 'deps=', decision['unresolved_dependencies'])
else:
    print('Run checkpoint 5 first, then rerun this cell.')

## 13. Run Checkpoint 6: Repair And Review Packaging

Checkpoint 6 converts verifier uncertainty into work items. It is not an approval layer. Every item from `review_queue.json` becomes a review packet and a repair attempt, but the attempt stays `review_only` until a future proof patch has field-level evidence and passes deterministic verification.

```bash
.venv/bin/python scripts/run_f1_checkpoint.py --checkpoint repair_review --city burnaby_r1
```

Outputs to inspect:

- `repair_attempts.json`: what the repair layer should try next.
- `review_packets.json`: what a human reviewer needs to answer.
- `repair_rerun_report.json`: whether any proof patch was rerun and promoted.

Safety rule: checkpoint 6 must keep verifier rerun attempts, guarded repair promotions, and false verified count at `0` unless a validated proof patch exists.


In [ ]:
repair_report_path = Path('outputs/f1_runs/burnaby_r1/checkpoint_6_repair_review/repair_rerun_report.json')
repair_attempts_path = Path('outputs/f1_runs/burnaby_r1/checkpoint_6_repair_review/repair_attempts.json')
review_packets_path = Path('outputs/f1_runs/burnaby_r1/checkpoint_6_repair_review/review_packets.json')
if repair_report_path.exists() and repair_attempts_path.exists() and review_packets_path.exists():
    repair_report = json.loads(repair_report_path.read_text())
    repair_attempts = json.loads(repair_attempts_path.read_text())
    review_packets = json.loads(review_packets_path.read_text())
    rows = [
        ('Repair attempts', len(repair_attempts)),
        ('Review packets', len(review_packets)),
        ('Verifier rerun attempts', repair_report['attempt_count']),
        ('Guarded promotions', repair_report['guarded_repair_promotion_count']),
        ('False verified', repair_report['false_verified_count']),
    ]
    for label, value in rows:
        print(f'{label}: {value}')
    print('\nFirst repair attempt:')
    print(json.dumps(repair_attempts[0], indent=2) if repair_attempts else 'No repair attempts')
else:
    print('Run checkpoint 6 first, then rerun this cell.')


## 14. Run Checkpoint 7: Guarded Repair Rerun

Checkpoint 7 is the first real repair checkpoint. It creates narrow proof patches only when the existing cited quote can safely fill a missing field. The current deterministic patch type is `applies_to`: if the source quote plainly names what the rule applies to, F1 patches that field, reruns the verifier, and promotes only if the verifier returns no gaps, dependencies, or risk flags.

```bash
.venv/bin/python scripts/run_f1_checkpoint.py --checkpoint guarded_repair --city burnaby_r1
```

Measured Burnaby result after the release-contract guard: 5 eligible proof patches, 5 verifier reruns, 3 guarded repair promotions, 2 not-used rows, 1 remaining rejected rule, and 0 false verified.


In [ ]:
guarded_base = Path('outputs/f1_runs/burnaby_r1/checkpoint_7_guarded_repair')
report_path = guarded_base / 'repair_rerun_report.json'
verification_path = guarded_base / 'verification_report.json'
verified_path = guarded_base / 'verified_rules.json'
review_path = guarded_base / 'review_queue.json'
if all(path.exists() for path in [report_path, verification_path, verified_path, review_path]):
    repair_report = json.loads(report_path.read_text())
    verification_report = json.loads(verification_path.read_text())
    verified_rules = json.loads(verified_path.read_text())
    review_queue = json.loads(review_path.read_text())
    print('Repair rerun report:', repair_report)
    print('Verification report:', verification_report)
    print('\nRepaired rules:')
    for rule in verified_rules:
        print(rule['rule_id'], rule['rule_type'], rule['operator'], rule['value'], rule['unit'], 'applies_to=', rule['applies_to'])
    print('\nRemaining review/rejected items:')
    for item in review_queue:
        print(item['rule_id'], item['status'], item['support_gaps'])
else:
    print('Run checkpoint 7 first, then rerun this cell.')


## 15. Run Checkpoint 8: Verified-Only GIS And Compliance

Checkpoint 8 turns verifier-promoted rules into product-facing constraints. This is a projection layer, not a verification layer: it cannot create a rule, repair a rule, or approve from review evidence.

```bash
.venv/bin/python scripts/run_f1_checkpoint.py --checkpoint gis_compliance --city burnaby_r1
```

Measured Burnaby result after the release-contract guard: 3 rules entered `gis_rule_contract.json`, 3 review/rejected/not-used items were excluded, and generated smoke compliance produced 3 approved, 3 rejected, and 1 needs-review decisions with 0 false approvals.

Important: these smoke scenarios prove wiring and safety boundaries. They are not a substitute for benchmark proposal cases.


In [ ]:
gis_base = Path('outputs/f1_runs/burnaby_r1/checkpoint_8_gis_compliance')
contract_path = gis_base / 'gis_rule_contract.json'
compliance_report_path = gis_base / 'compliance_report.json'
if contract_path.exists() and compliance_report_path.exists():
    contract = json.loads(contract_path.read_text())
    compliance_report = json.loads(compliance_report_path.read_text())
    print('GIS contract rules:', contract['export_rule_count'])
    print('Excluded review/rejected:', contract['excluded_review_rule_count'])
    print('Compliance report:', compliance_report)
    print('\nGIS rows:')
    for rule in contract['rules']:
        print(rule['rule_id'], rule['rule_object'], rule['operator'], rule['value'], rule['unit'], rule['geometry_target'])
else:
    print('Run checkpoint 8 first, then rerun this cell.')


## 16. Run Checkpoint 9: Chatbot Evaluation Harness

Checkpoint 9 evaluates the chatbot boundary before adding a real model runtime. It reads F1 artifacts, answers only with cited evidence when it can answer, and refuses approval authority.

```bash
.venv/bin/python scripts/run_f1_checkpoint.py --checkpoint chatbot_eval --city burnaby_r1
```

Measured Burnaby result: 3 eval cases, 2 cited answers, 1 uncertain/refusal answer, 0 approval claims, and 0 false approvals. The artifact says `model_runtime_used=false`; this is the harness that GPT-OSS-120B + RAG must later pass.


In [ ]:
chatbot_path = Path('outputs/f1_runs/burnaby_r1/checkpoint_9_chatbot_eval/chatbot_eval.json')
if chatbot_path.exists():
    chatbot_eval = json.loads(chatbot_path.read_text())
    print('Metrics:', chatbot_eval['metrics'])
    for case in chatbot_eval['cases']:
        answer = case['answer']
        print('\nQ:', case['question'])
        print('A:', answer['answer'])
        print('can_answer:', answer['can_answer'])
        print('citations:', answer['cited_evidence_ids'])
else:
    print('Run checkpoint 9 first, then rerun this cell.')


## 17. Run Checkpoint 10: Command Center Preview

Checkpoint 10 turns F1 artifacts into a product-facing command-center preview. It writes a JSON manifest and a self-contained HTML file.

```bash
.venv/bin/python scripts/run_f1_checkpoint.py --checkpoint command_center --city burnaby_r1
```

Open this file in a browser:

```text
outputs/f1_runs/burnaby_r1/checkpoint_10_command_center/command_center_preview.html
```

Measured Burnaby result after the latest extraction and compliance improvements: 8 dashboard cards, 38 verified GIS rules, 13 review/rejected/not-used items, compliance status visualization, and chatbot safety metrics. This is still a preview, not the final interactive app.


In [ ]:
command_manifest_path = Path('outputs/f1_runs/burnaby_r1/checkpoint_10_command_center/command_center_manifest.json')
if command_manifest_path.exists():
    manifest = json.loads(command_manifest_path.read_text())
    print('Cards:')
    for card in manifest['cards']:
        print(card['label'], card['value'])
    print('\nVerified rules shown:', len(manifest['verified_rules']))
    print('Review items shown:', len(manifest['review_queue']))
    print('Preview file: outputs/f1_runs/burnaby_r1/checkpoint_10_command_center/command_center_preview.html')
else:
    print('Run checkpoint 10 first, then rerun this cell.')


## 18. Run Checkpoint 11: Real GPT-OSS Chatbot Eval

Checkpoint 11 runs the real model-backed chatbot eval through OpenRouter with `openai/gpt-oss-120b`. It uses bounded F1 context and deterministic safety guardrails around the model.

```bash
.venv/bin/python scripts/run_f1_checkpoint.py --checkpoint model_chatbot_eval --city burnaby_r1 --chat-model openai/gpt-oss-120b
```

Measured Burnaby result: 3 model eval cases, 2 cited answers, 0 citation-missing answers, 1 refusal/uncertain answer, 2 model-handled answers, 1 safety-guard answer, 0 approval claims, and 0 false approvals.

Key lesson: a real LLM can produce useful natural language, but F1 should still route approval and policy-boundary requests through deterministic guardrails.


In [ ]:
model_chat_path = Path('outputs/f1_runs/burnaby_r1/checkpoint_11_model_chatbot_eval/chatbot_eval.json')
if model_chat_path.exists():
    model_chat = json.loads(model_chat_path.read_text())
    print('Runtime:', model_chat['chatbot'])
    print('Metrics:', model_chat['metrics'])
    for case in model_chat['cases']:
        answer = case['answer']
        print('\nQ:', case['question'])
        print('can_answer:', answer['can_answer'])
        print('citations:', answer['cited_evidence_ids'])
        print('A:', answer['answer'][:500])
else:
    print('Run checkpoint 11 first, then rerun this cell.')


## 19. Run Checkpoint 12: Real GPT-OSS Red-Team Chatbot Eval

Checkpoint 12 runs a harder real model-backed chatbot test through OpenRouter with `openai/gpt-oss-120b`. It tests prompt injection, rejected-rule pressure, follow-up wording, citation-specific questions, verified-rule inventory lookup, and approval-boundary pressure.

```bash
.venv/bin/python scripts/run_f1_checkpoint.py --checkpoint model_chatbot_redteam --city burnaby_r1 --chat-model openai/gpt-oss-120b
```

Measured Burnaby result: 6 red-team cases, 6 passed, 0 failed, 0 citation-missing, 0 refusal-missing, 1 model-handled answer, 5 safety-guard answers, 0 approval claims, and 0 false approvals.

Key lesson: red-team eval is not just "did the model answer." A paid-worthy command center needs routing judgment: structured inventory questions and unsafe status-changing requests should be deterministic, while open explanatory questions can use the LLM with citation checks.


In [ ]:
redteam_path = Path('outputs/f1_runs/burnaby_r1/checkpoint_12_model_chatbot_redteam/chatbot_redteam_eval.json')
if redteam_path.exists():
    redteam = json.loads(redteam_path.read_text())
    print('Runtime:', redteam['chatbot'])
    print('Metrics:', redteam['metrics'])
    print('\nCases:')
    for case in redteam['cases']:
        print(case['case_id'], 'passed=', case['passed'], 'citations=', case['cited_evidence_ids'])
        if not case['passed']:
            print('  answer:', case['raw_answer'][:500])
else:
    print('Run checkpoint 12 first, then rerun this cell.')


## 20. Run Checkpoint 13: Multi-City Generalization

Checkpoint 13 runs the same F1 artifact pipeline across Burnaby, Vancouver, Calgary, and Surrey. The point is not to claim all cities are release-ready. The point is to measure whether the product structure generalizes while keeping the safety boundary intact.

```bash
.venv/bin/python scripts/run_f1_checkpoint.py --checkpoint multi_city --multi-cities burnaby_r1,vancouver_rs,calgary_rcg,surrey_r3 --no-docling
```

Aggregate output:

```text
outputs/f1_runs/multi_city/checkpoint_13_multi_city/f1_multi_city_report.json
outputs/f1_runs/multi_city/checkpoint_13_multi_city/f1_multi_city_report.md
outputs/f1_runs/multi_city/checkpoint_13_multi_city/f1_summary.json
```

Measured result from that checkpoint-13 run: 4 cities completed, 0 failed, 1 release-safe benchmark city, 3 partial/exploratory cities, 0 false verified, and 0 false approvals. That historical run exported 3 Burnaby GIS rules and kept 3 items out of verified output; later Burnaby checkpoints improved coverage substantially.

Key lesson: multi-city coverage is not the same as multi-city release readiness. Burnaby has gold rules and proposal cases. Vancouver and Calgary have gold rules but no proposal cases. Surrey has no gold benchmark yet, so it is exploratory only.



In [ ]:
multi_city_path = Path('outputs/f1_runs/multi_city/checkpoint_13_multi_city/f1_multi_city_report.json')
if multi_city_path.exists():
    import json
    report = json.loads(multi_city_path.read_text())
    print('Completed cities:', report['completed_city_count'])
    print('Failed cities:', report['failed_city_count'])
    print('False verified total:', report['false_verified_total'])
    print('False approval total:', report['false_approval_total'])
    print('Runtime seconds:', report['runtime_seconds'])
    print('\nCity summary:')
    for city in report['cities']:
        print(
            city['city_key'],
            city['release_label'],
            'candidates=', city.get('candidate_count'),
            'verified=', city.get('verified_count'),
            'review=', city.get('review_queue_count'),
            'gis=', city.get('gis_export_rule_count'),
        )
else:
    print('Run checkpoint 13 first, then rerun this cell.')


## 21. Run Checkpoint 14: Benchmark Evaluation

Checkpoint 14 runs the full F1 artifact flow and then evaluates the result against gold rules and proposal cases. This is the checkpoint that prevents us from confusing a clean smoke run with release readiness.

```bash
.venv/bin/python scripts/run_f1_checkpoint.py --checkpoint benchmark_eval --city burnaby_r1
```

Output:

```text
outputs/f1_runs/burnaby_r1/checkpoint_14_benchmark_eval/f1_benchmark_report.json
outputs/f1_runs/burnaby_r1/checkpoint_14_benchmark_eval/f1_benchmark_report.md
outputs/f1_runs/burnaby_r1/checkpoint_14_benchmark_eval/f1_summary.json
```

Measured Burnaby result after proof-aware retrieval, table-aware extraction, guarded repair carry-forward, and stricter compliance review routing: verified precision 1.0, false verified 0, false approvals 0, GIS review leaks 0, retrieval recall at 6 = 0.95, candidate recall = 0.975, verified recall = 0.85, proposal decision accuracy = 1.0, and proposal case accuracy = 1.0. Current quality gates pass.

Key lesson: F1 improved because we added better evidence and candidate coverage, not because we loosened verification. The remaining review queue is still intentional: uncertain rules stay non-executable until proof or human review resolves them.


In [ ]:
benchmark_path = Path('outputs/f1_runs/burnaby_r1/checkpoint_14_benchmark_eval/f1_benchmark_report.json')
if benchmark_path.exists():
    import json
    report = json.loads(benchmark_path.read_text())
    print('Release gates passed:', report['quality_gates']['passed'])
    print('Failed gates:', report['quality_gates']['failed_gates'])
    print('Retrieval:', report['retrieval_benchmark']['metrics'])
    print('Rules:', {k: report['rule_metrics'][k] for k in ['candidate_recall', 'verified_precision', 'verified_recall', 'false_verified_count']})
    print('GIS:', report['gis_contract_metrics'])
    print('Proposal:', report['proposal_metrics'])
else:
    print('Run checkpoint 14 first, then rerun this cell.')


## 22. Run Checkpoint 15: Interactive Command Center App

Checkpoint 15 turns the measured F1 artifact set into the first interactive command-center product surface. It is still a self-contained HTML file, but it behaves like an app: searchable verified rules, review queue, evidence inspector, benchmark safety readout, chatbot boundary metrics, and a single-rule compliance sandbox.

```bash
.venv/bin/python scripts/run_f1_checkpoint.py --checkpoint command_center_app --city burnaby_r1
```

Open this file in a browser:

```text
outputs/f1_runs/burnaby_r1/checkpoint_15_command_center_app/command_center_app.html
```

Measured Burnaby result: 38 verified rules, 13 review items, 77 compliance decisions, 9 command cards, 5 product workflows, all current benchmark quality gates passed, false verified = 0, and false approvals = 0.

Key lesson: this is the first F1 checkpoint that starts to feel like product software rather than backend research output. It still stays read-only over artifacts, so product usability improves without weakening the verifier boundary.


In [ ]:
app_manifest_path = Path('outputs/f1_runs/burnaby_r1/checkpoint_15_command_center_app/command_center_app_manifest.json')
if app_manifest_path.exists():
    import json
    manifest = json.loads(app_manifest_path.read_text())
    print('Workflow health:', manifest['workflow_health'])
    print('Cards:', len(manifest['cards']))
    print('Workflows:', [workflow['id'] for workflow in manifest['product_workflows']])
    print('Verified rules:', len(manifest['verified_rules']))
    print('Review items:', len(manifest['review_queue']))
    print('App file: outputs/f1_runs/burnaby_r1/checkpoint_15_command_center_app/command_center_app.html')
else:
    print('Run checkpoint 15 first, then rerun this cell.')


## 24. Run The F1 Streamlit Product UI

The static checkpoint 15 HTML proved the product shape. The current Streamlit app is the served product UI for day-to-day use.

```bash
.venv/bin/python -m streamlit run dashboard/f1_command_center.py --server.port 8631
```

Open:

```text
http://localhost:8631
```

The app reads `outputs/f1_runs/burnaby_r1/checkpoint_15_command_center_app/command_center_app_manifest.json` by default. It is not an M8/M8.5 dashboard. It is the F1 product surface, but it now borrows the proven M7 civic-console interaction pattern.

The active navigation is:

| View | Purpose | Safety boundary |
|---|---|---|
| 0 Results + Chat | Show the candidate check flow, ask source-grounded questions, show guardrails, and guide next actions. | Chat cannot approve or edit rules. |
| 1 Source Documents | Inspect official PDF provenance, source spans, and retrieval evidence packets. | Source is ground truth, not a verifier decision. |
| 2 Verified Rules | Browse grouped verified rules by family and source page. | Uses verified rules only. |
| 3 Human Review | Inspect one blocked candidate, why it is held, and how it differs from a verified rule. | Review items cannot execute. |
| 4 Repair Evidence | Show shadow repair/rerun results and guarded repair promotions. | Repair does not directly promote. |
| 5 GIS Handoff | Read and download verified-only map-ready constraints. | GIS consumes verified-only rules. |
| 6 Quality Audit | Review precision, recall, source support, and gap diagnostics. | Benchmarks decide release confidence. |

Why the old M8 page disappeared: Streamlit automatically exposes files under `dashboard/pages/`. Those pages were research viewers, so F1 archives them under `archive/m_series/dashboard_pages/` and keeps the active product navigation F1-only.

Measured validation after the reference-style dashboard update: HTTP 200, `_stcore/health = ok`, browser smoke across all seven pages, 38 verified rules loaded, 13 review items loaded, and 709 tests plus 3 subtests passed.

Key lesson: a product UI can reuse the strongest M-series interaction pattern without becoming an M-series product. F1 still reads canonical F1 artifacts and keeps the verifier as the only promotion authority.


In [ ]:
from dashboard.f1_data import load_f1_output, benchmark_gate_rows, rule_rows, review_rows, review_gap_rows
from dashboard.f1_command_center import NAV_ITEMS, _candidate_flow_counts

data = load_f1_output()
print('Navigation:', NAV_ITEMS)
print('Candidate flow:', _candidate_flow_counts(data))
print('Benchmark gates:', benchmark_gate_rows(data))
print('Verified rule rows:', len(rule_rows(data)))
print('Review rows:', len(review_rows(data)))
print('Review gaps:', review_gap_rows(data))
print('Run UI: .venv/bin/python -m streamlit run dashboard/f1_command_center.py --server.port 8631')


## 26. Active-Learning Checkpoint

Before upgrading F1, answer these without looking back:

1. Why is RAG not allowed to verify a rule?
2. What must a promoted rule have?
3. Why is a city without gold benchmark exploratory only?
4. Why does repair send candidate copies back to verification?
5. What would make a chatbot answer unsafe?
6. What artifact should GIS consume?
7. What should be deleted or simplified before adding another model?
8. Why does F1 verify a source hash before parsing the PDF?
9. Why are source spans still not verified rules?
10. Why is a retrieval hit not enough to approve a rule?
11. Why should checkpoint 4 candidates stay `review_needed`?
12. Why is `context_required` safer than forcing a weak candidate into `verified`?
13. Why does checkpoint 6 keep repair attempts review-only?
14. What evidence would a future repair patch need before rerunning verification?
15. Why is guarded repair different from repair confidence?
16. Why did the operator-unsupported rule remain rejected after checkpoint 7?
17. Why must GIS read only verified rule projections?
18. Why is a smoke compliance scenario not a release benchmark?
19. Why must the chatbot refuse approval requests?
20. Why must the model-backed chatbot pass the deterministic harness first?
21. Why should the command center read canonical F1 artifacts instead of scattered old outputs?
22. Why is the F1 Streamlit app not allowed to expose M8/M8.5 pages?
23. Why is the Review Desk useful even though it cannot promote rules?
24. What should the Compliance Lab say when all selected checks pass?
25. Which view should a planner use to inspect the evidence behind one verified rule?
26. Which view should a reviewer use to understand why a candidate is blocked?
27. Why is real model output still not allowed to write verifier decisions?
28. What failure should force a prompt/runtime rollback?
29. Why is a multi-city run not automatically a multi-city release?
30. What benchmark evidence is missing for Surrey?
31. Why should non-Burnaby zero verified rules be treated as a coverage signal instead of a safety failure?
32. Why can a proof-passed rule still become `not_used`?
33. Which checkpoint proves verified precision against gold rules?
34. Why is candidate recall now the next upgrade target?
35. Why would relaxing the verifier be the wrong fix for low proposal accuracy?

Write your answers below before starting the next checkpoint.


## 27. Update Log

| Date | Checkpoint | What changed | Evidence |
|---|---|---|---|
| 2026-07-05 | F1 checkpoint 0 | Created F1 constitution, contracts, skeleton, archive index, and living textbook | `scripts/run_f1_checkpoint.py` and `f1_summary.json` |
| 2026-07-05 | F1 checkpoint 1 | Added source registry/provenance validation with SHA-256 checks | `source_registry.json` and `checkpoint_report.md` |
| 2026-07-05 | F1 checkpoint 2 | Added Docling-first document intelligence with fallback parsing | `document_blocks.json`, `source_spans.json`, and `document_intelligence_report.json` |
| 2026-07-05 | F1 checkpoint 3 | Added deterministic proof-aware retrieval smoke layer | `retrieval_report.json` |
| 2026-07-05 | F1 checkpoint 4 | Added conservative structured extraction into review-needed candidates | `candidate_rules.json` and `extraction_report.json` |
| 2026-07-05 | F1 checkpoint 5 | Added deterministic fail-closed verification outputs | `verification_decisions.json`, `verified_rules.json`, `review_queue.json`, and `verification_report.json` |
| 2026-07-06 | F1 checkpoint 6 | Added repair/review worklist packaging without direct promotion | `repair_attempts.json`, `review_packets.json`, and `repair_rerun_report.json` |
| 2026-07-06 | F1 checkpoint 7 | Added deterministic proof-patch reruns with guarded verifier promotions | `verifier_rerun_requests.json`, `guarded_repair_promotions.json`, and updated `verified_rules.json` |
| 2026-07-06 | F1 checkpoint 8 | Added verified-only GIS contract and tri-state compliance smoke outputs | `gis_rule_contract.json`, `compliance_decisions.json`, and `compliance_report.json` |
| 2026-07-06 | F1 checkpoint 9 | Added source-cited chatbot safety evaluation harness | `chatbot_eval.json` |
| 2026-07-06 | F1 checkpoint 10 | Added self-contained command-center preview over F1 artifacts | `command_center_manifest.json` and `command_center_preview.html` |
| 2026-07-06 | F1 checkpoint 11 | Ran real GPT-OSS-120B chatbot eval through OpenRouter with citation and approval-boundary checks | `outputs/f1_runs/burnaby_r1/checkpoint_11_model_chatbot_eval/chatbot_eval.json` |
| 2026-07-06 | F1 checkpoint 12 | Added deterministic chatbot safety guardrails and passed real GPT-OSS-120B red-team/follow-up eval | `outputs/f1_runs/burnaby_r1/checkpoint_12_model_chatbot_redteam/chatbot_redteam_eval.json` |
| 2026-07-06 | F1 checkpoint 13 | Ran multi-city generalization across Burnaby, Vancouver, Calgary, and Surrey with aggregate safety reporting | `outputs/f1_runs/multi_city/checkpoint_13_multi_city/f1_multi_city_report.json` |
| 2026-07-06 | F1 checkpoint 14 | Improved proof-aware retrieval, table-aware extraction, guarded repair carry-forward, and compliance review routing; all current Burnaby benchmark gates pass | `outputs/f1_runs/burnaby_r1/checkpoint_14_benchmark_eval/f1_benchmark_report.json` |
| 2026-07-06 | F1 checkpoint 15 | Added interactive command-center app with rule explorer, review workbench, evidence inspector, compliance sandbox, and safety panels | `outputs/f1_runs/burnaby_r1/checkpoint_15_command_center_app/command_center_app.html` |
| 2026-07-06 | F1 Streamlit UI | Added served Streamlit product UI over checkpoint 15 artifacts | `dashboard/f1_command_center.py` |
| 2026-07-06 | F1 product UI redesign | Archived old M8/M8.5 Streamlit pages and rebuilt the served dashboard as a left-rail F1 command center | `dashboard/f1_command_center.py` and `archive/m_series/dashboard_pages/` |
| 2026-07-06 | F1 reference-style dashboard | Reworked the served F1 UI to match the proven M7 civic-console layout: BV rail, candidate flow, source docs, verified rules, human review, repair evidence, GIS handoff, and quality audit | `dashboard/f1_command_center.py` |


## 28. Checkpoint 16: Product Operations Layer

Checkpoint 16 moves F1 from a readable command-center preview into a usable operator surface.
The important boundary is unchanged: operator actions are drafts, not legal decisions.
They write to `operator_review_actions.json` only and do not edit `verified_rules.json`,
`verification_decisions.json`, or `gis_rule_contract.json`.

New checkpoint 16 artifacts:

| Artifact | Purpose | Authority |
|---|---|---|
| `operator_action_templates.json` | Recommended next-step templates for every review item | Advisory only |
| `operator_review_actions.json` | Append-only local review notes saved from the UI | No promotion authority |
| `scenario_templates.json` | Numeric verified-rule examples for scenario checks | Scoped rule checks only |
| `evidence_navigation.json` | Rule-to-source navigation index for verified and review items | Evidence navigation |
| `operations_report.json` | Counts and safety flags for the operations layer | Audit summary |

Measured Burnaby checkpoint 16:

| Metric | Value |
|---|---:|
| Verified rules | 38 |
| Review items | 13 |
| Review action templates | 13 |
| Scenario templates | 38 |
| Evidence navigation rows | 77 |
| Retrieval recall at 6 | 0.95 |
| Candidate recall | 0.975 |
| Verified precision | 1.0 |
| Verified recall | 0.85 |
| False verified | 0 |
| False approvals | 0 |


In [ ]:
from pathlib import Path
import json

base = Path('outputs/f1_runs/burnaby_r1/checkpoint_16_product_operations')
summary = json.loads((base / 'f1_summary.json').read_text())
manifest = json.loads((base / 'command_center_app_manifest.json').read_text())
ops = json.loads((base / 'operations_report.json').read_text())
benchmark = json.loads((base / 'f1_benchmark_report.json').read_text())

print('Checkpoint:', summary['checkpoint'])
print('App version:', manifest['app_version'])
print('Workflow health:', manifest['workflow_health'])
print('Operations:', {
    'action_templates': ops['action_template_count'],
    'scenario_templates': ops['scenario_template_count'],
    'evidence_navigation_rows': ops['evidence_navigation_count'],
    'writes_rule_status': ops['writes_rule_status'],
    'writes_gis_contract': ops['writes_gis_contract'],
})
print('Benchmark:', {
    'quality_gates_passed': benchmark['quality_gates']['passed'],
    'retrieval_recall_at_6': benchmark['retrieval_benchmark']['metrics']['retrieval_recall_at_6'],
    'candidate_recall': benchmark['rule_metrics']['candidate_recall'],
    'verified_precision': benchmark['rule_metrics']['verified_precision'],
    'false_verified_count': benchmark['rule_metrics']['false_verified_count'],
    'false_approval_count': benchmark['proposal_metrics']['false_approval_count'],
})


### Active-Learning Checkpoint 16 Questions

Answer these before F1.5/F2 planning:

1. Why is `operator_review_actions.json` allowed to persist notes but not allowed to change rule status?
2. Which artifact should a GIS system consume, and why should it ignore review items?
3. What is the difference between a scenario check passing one verified rule and a real permit approval?
4. Why does checkpoint 16 still preserve `false_verified_count = 0` even though it adds operator actions?
5. If a reviewer saves a note saying "ready for repair", what must happen before the rule becomes verified?


## 29. Checkpoint 16 Finalization: Product UI, Chat, And Safety Audit

The final Checkpoint 16 pass asks a product question: would a first user understand what F1 does, trust the safety boundary, and know where to go next?

The implementation answer is now stronger than the previous prototype:

| Area | Before final pass | After final pass |
|---|---|---|
| First screen | Functional artifact browser | Civic command-center landing surface with benchmark status |
| Chat answers | Plain text summaries | Direct answer, supporting facts, next step, and citations |
| Chat safety | Deterministic refusal existed but was less explicit | Approval, permit, promotion, and legality requests are clearly blocked |
| Rule matching | Token matching could confuse related phrases | Phrase-aware matching separates `lot area`, `lot coverage`, and height queries |
| User journey | Many tabs, weaker priority | Overview, verified rules, review, GIS, chat, and quality audit are framed as one workflow |
| Documentation | Checkpoint 16 operations documented | Final readiness report and live screenshot audit added |

The important product commandment is unchanged: the chat layer can explain and navigate, but it cannot approve, promote, or edit verifier-owned artifacts.


In [ ]:
from pathlib import Path
import json

run = Path('outputs/f1_runs/burnaby_r1/checkpoint_16_product_operations')
report = Path('docs/F1_FINAL_READINESS_REPORT.md')
audit_dir = Path('docs/f1_product_audit_2026_07_06')
summary = json.loads((run / 'f1_summary.json').read_text())
benchmark = json.loads((run / 'f1_benchmark_report.json').read_text())
ops = json.loads((run / 'operations_report.json').read_text())

readiness_snapshot = {
    'checkpoint': summary['checkpoint'],
    'quality_gates_passed': benchmark['quality_gates']['passed'],
    'verified_rules': summary['verified_rule_count'],
    'review_items': summary['review_queue_count'],
    'verified_precision': benchmark['rule_metrics']['verified_precision'],
    'retrieval_recall_at_6': benchmark['retrieval_benchmark']['metrics']['retrieval_recall_at_6'],
    'candidate_recall': benchmark['rule_metrics']['candidate_recall'],
    'false_verified_count': benchmark['rule_metrics']['false_verified_count'],
    'false_approval_count': benchmark['proposal_metrics']['false_approval_count'],
    'operations_write_rule_status': ops['writes_rule_status'],
    'operations_write_gis_contract': ops['writes_gis_contract'],
    'readiness_report_exists': report.exists(),
    'audit_screenshots': sorted(p.name for p in audit_dir.glob('*.png')),
}
readiness_snapshot


### Visual Audit Trail

The final pass captured live screenshots from the running Streamlit app:

| Screenshot | What it checks |
|---|---|
| `docs/f1_product_audit_2026_07_06/01-overview-chat.png` | First screen, status strip, and chat answer presentation |
| `docs/f1_product_audit_2026_07_06/02-human-review.png` | Review workbench and non-promoting action framing |
| `docs/f1_product_audit_2026_07_06/03-gis-handoff.png` | Verified-only GIS/compliance handoff |
| `docs/f1_product_audit_2026_07_06/04-quality-audit.png` | Benchmark and safety audit surface |

Use these images as a product regression reference before F1.5 or F2 redesign work.


### Active-Learning Finalization Questions

Before the next upgrade, answer these in your own words:

1. Why is a good-looking dashboard still unsafe if it can mutate `verified_rules.json` directly?
2. Why should the chatbot say "I cannot approve" even when it has enough information to explain a rule?
3. What is the practical difference between `verified_precision = 1.0` and `verified_recall = 0.85`?
4. Which F1 artifact should a future parcel/GIS engine consume first, and why?
5. What evidence would prove that a future live LLM chatbot improves user understanding without weakening safety?



## 30. Checkpoint 17: Cloud Product UI Polish

Checkpoint 17 is a product-readiness UI pass for the Streamlit Cloud surface. The goal is not to add new verification authority; it is to make the existing F1 artifacts feel like a product a planner, reviewer, or GIS user can understand quickly.

### What Changed

| Product surface | Before | After |
|---|---|---|
| Navigation | Research-style labels such as Results + Chat and Human Review | F1 product labels: Command Center, Sources, Review Workbench, GIS + Compliance |
| First screen | Useful but report-like | Source-to-decision command center with benchmark-safe status, verified counts, review queue, and verifier flow |
| Count language | Review counts could look inconsistent | Header now separates operator queue records from candidates still needing proof |
| Chat answer | Artifact summary card | Answer-first response card with meaning, next step, and citations |
| Visual polish | Default Streamlit feel and overly technical typography | Product-grade typography, spacing, status chips, cards, and guardrail layout |

### Product Boundary Preserved

The UI is prettier, but the authority model did not change:

```mermaid
flowchart LR
    A[Official source PDF] --> B[Evidence retrieval]
    B --> C[Candidate rule extraction]
    C --> D[Deterministic verifier]
    D -->|verified only| E[GIS + compliance]
    D -->|uncertain| F[Review Workbench]
    B --> G[Source-grounded chat]
    G -->|explain only| H[User understanding]
```

The chatbot still cannot approve, verify, or promote. GIS and compliance still consume verified-only rules.

### Validation Snapshot

| Check | Result |
|---|---|
| Local Streamlit health | `ok` from `/_stcore/health` |
| Focused dashboard tests | `3 passed` |
| Syntax check | `py_compile` passed for dashboard and entrypoint files |
| Browser QA | Local F1 app rendered with F1-only navigation and source-grounded chat response |

### Execution Commands

```bash
/Users/thomas/Documents/Capstone_prototype/Burnaby_prototype/.venv/bin/python -m py_compile dashboard/f1_command_center.py dashboard/f1_data.py streamlit_app.py
/Users/thomas/Documents/Capstone_prototype/Burnaby_prototype/.venv/bin/python -m pytest -q tests/test_f1_streamlit_dashboard.py -q
curl -sS http://localhost:8651/_stcore/health
```


In [ ]:

from pathlib import Path
import json

run = Path('outputs/f1_runs/burnaby_r1/checkpoint_16_product_operations')
manifest = json.loads((run / 'command_center_app_manifest.json').read_text())
verification = json.loads((run / 'verification_report.json').read_text())

snapshot = {
    'city': manifest.get('city_key'),
    'verified_rules': manifest.get('workflow_health', {}).get('verified_rule_count'),
    'operator_queue_records': manifest.get('workflow_health', {}).get('review_queue_count'),
    'still_needs_proof': verification.get('status_counts', {}).get('context_required', 0) + verification.get('status_counts', {}).get('not_used', 0),
    'rejected': verification.get('status_counts', {}).get('rejected', 0),
    'verified_precision': manifest.get('benchmark_metrics', {}).get('verified_precision'),
    'false_verified': manifest.get('workflow_health', {}).get('false_verified_count'),
    'false_approvals': manifest.get('workflow_health', {}).get('false_approval_count'),
}

snapshot



### Active-Learning Checkpoint 17 Questions

1. Why is `operator queue` different from `still needs proof` in the F1 dashboard?
2. Why is it safer for the chatbot to explain GIS boundaries than to answer with permit approval language?
3. Which UI elements show that verification, not repair or chat, is the promotion authority?
4. What would make this Streamlit surface feel more like a paid product without weakening the verifier boundary?
5. Before F1.5, which one product workflow should be improved first: source inspection, review workbench, GIS scenario checks, or chatbot follow-up memory?
